# UARP on Colab
Runs the utility-aware retention experiments on SKAB (and optionally Pump Sensor).
Run the cells top to bottom. Runtime: CPU is enough (full run ~10-25 min, `--fast` ~1 min).

## 1. Get the code
Upload `uarp_project.zip` when prompted.

In [ ]:
from google.colab import files
up = files.upload()            # choose uarp_project.zip
!unzip -q -o uarp_project.zip
%cd uarp_project
!pip -q install -r requirements.txt

## 2. Add the dataset
**Option A (easiest): download in your browser from Kaggle, then upload here.**
Unzip is automatic. For SKAB the files end up under `data/skab/`.

In [ ]:
import zipfile, os
up = files.upload()            # choose the Kaggle SKAB zip you downloaded
os.makedirs("data/skab", exist_ok=True)
for name in up:
    if name.endswith(".zip"):
        zipfile.ZipFile(name).extractall("data/skab")
print("done")

**Option B: Kaggle API** (skip if Option A worked). Upload your `kaggle.json` when asked.
If the download fails, the dataset slug has changed: use Option A.

In [ ]:
# from google.colab import files; files.upload()      # kaggle.json
# !mkdir -p ~/.kaggle && mv kaggle.json ~/.kaggle/ && chmod 600 ~/.kaggle/kaggle.json
# !pip -q install kaggle
# !kaggle datasets download -d yuriykatser/skoltech-anomaly-benchmark-skab -p data/skab --unzip

**Optional second dataset (Pump Sensor Data):** upload `sensor.csv` to `data/pump/`, then set `DATASETS = ["skab", "pump"]` in `config.py` or pass `--dataset skab pump` below.

In [ ]:
# import os; os.makedirs("data/pump", exist_ok=True)
# up = files.upload()      # choose sensor.csv
# import shutil; shutil.move("sensor.csv", "data/pump/sensor.csv")

In [ ]:
!find data -name '*.csv' | head -5; echo; find data -name '*.csv' | wc -l

## 3. Smoke test, then the full run

In [ ]:
!python run_all.py --fast

In [ ]:
!python run_all.py

## 4. Look at the results

In [ ]:
import pandas as pd, glob
from IPython.display import Image, display
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
for d in ["skab"]:
    m = pd.read_csv(f"results/{d}/metrics_main.csv")
    cols = ["policy"] + [c for c in m.columns if c.endswith("_mean") and c.split("_mean")[0] in
            ("anomaly_recall","event_coverage","wasserstein","if_auprc","rf_auprc","z_f1")]
    display(m[cols].round(3))
    print(open(f"results/{d}/RESULTS_SUMMARY.txt").read())
    for f in sorted(glob.glob(f"results/{d}/fig_*.png")):
        display(Image(f, width=800))

## 5. Download everything

In [ ]:
!zip -q -r results.zip results
files.download("results.zip")